# Forza steering training
Start with the CPU synthetic smoke path below. For real training, select a GPU runtime and set `SYNTHETIC = False`. All training uses the same package CLI.

**Repository setup:** while signed into private GitHub, select the branch containing this pipeline and choose **Code → Download ZIP**. The upload cell below accepts that ZIP and installs it. Only upload repository code you trust. No token is stored in this notebook. Alternatively, securely clone outside this notebook, set `REPO_SOURCE = 'existing'`, and set `EXISTING_REPO`.


In [ ]:
from pathlib import Path
REPO_SOURCE = 'upload_zip'  # or 'existing'
EXISTING_REPO = Path('/content/stormhacks')
SYNTHETIC = True
DEVICE = 'cpu' if SYNTHETIC else 'cuda'
EPOCHS = 2 if SYNTHETIC else 10
BATCH_SIZE = 4 if SYNTHETIC else 64
RESUME = False  # True resumes RUN/last.pt to EPOCHS total epochs.
DATA = Path('/content/forza-synthetic' if SYNTHETIC else '/content/forza-data')
REUSE_DATA = False  # True only after a successful extraction/generation in this runtime.


The next cell provides shared ZIP containment and session-layout checks. It is copied from `scripts/colab_archives.py` so repository upload needs no preinstalled project package. Existing destinations are never overwritten.


In [ ]:
"""Stdlib-only Colab ZIP helpers, embedded in the notebook for bootstrap use.

Keep the notebook's archive-helpers cell in sync with this file. Only extract
archives whose code/data you trust; containment does not establish provenance.
"""
from pathlib import Path, PurePosixPath
import json
import shutil
import stat
import tempfile
import zipfile


def extract_zip(archive, destination):
    """Extract to a new directory; reject traversal, links, devices and collisions."""
    destination = Path(destination)
    if destination.exists():
        raise ValueError(f'extraction destination already exists: {destination}')
    with zipfile.ZipFile(archive) as source:
        entries, seen = [], set()
        for member in source.infolist():
            name = member.filename
            relative = PurePosixPath(name)
            mode = member.external_attr >> 16
            kind = stat.S_IFMT(mode)
            if (not name or '\\' in name or relative.is_absolute()
                    or '..' in relative.parts or ':' in name or not relative.parts):
                raise ValueError(f'unsafe ZIP path: {name!r}')
            if kind not in (0, stat.S_IFREG, stat.S_IFDIR):
                raise ValueError(f'ZIP links/special files are forbidden: {name!r}')
            target = destination.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(destination.resolve()):
                raise ValueError(f'ZIP path escapes destination: {name!r}')
            key = str(relative).casefold()
            if key in seen:
                raise ValueError(f'duplicate ZIP path: {name!r}')
            seen.add(key)
            entries.append((member, target))
        destination.mkdir(parents=True, exist_ok=False)
        try:
            for member, target in entries:
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with source.open(member) as incoming, target.open('xb') as outgoing:
                        shutil.copyfileobj(incoming, outgoing)
        except BaseException:
            shutil.rmtree(destination)
            raise
    return destination


def find_repository(extracted):
    """Accept a repository ZIP with either no wrapper or one GitHub wrapper."""
    extracted = Path(extracted)
    candidates = [extracted] + [p for p in extracted.iterdir() if p.is_dir()]
    candidates = [p for p in candidates if (p / 'pyproject.toml').is_file()
                  and (p / 'src/forza_ai/training/cli.py').is_file()]
    if len(candidates) != 1:
        raise ValueError('ZIP must contain exactly one repository with the training pipeline')
    return candidates[0]


def extract_sessions(archives, destination):
    """Transactionally unpack completed session ZIPs and validate every session.

    Each ZIP holds one session at its root, or session folders immediately below
    its root. Extraction happens on the destination filesystem, not Google Drive.
    """
    from forza_ai.data.sessions import load_sessions

    destination = Path(destination)
    archives = list(archives)
    if not archives or destination.exists():
        raise ValueError('provide at least one ZIP and a new destination directory')
    destination.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='forza-unpack-', dir=destination.parent) as temporary:
        staging = Path(temporary)
        combined = staging / 'sessions'
        combined.mkdir()
        for index, archive in enumerate(archives):
            unpacked = extract_zip(archive, staging / f'archive-{index}')
            if (unpacked / 'metadata.json').is_file():
                candidates = [unpacked]
            else:
                candidates = list(unpacked.iterdir())
                if not candidates or any(not p.is_dir() for p in candidates):
                    raise ValueError(f'{archive}: expected session folders immediately below ZIP root')
            for candidate in candidates:
                manifest = candidate / 'metadata.json'
                if not manifest.is_file():
                    raise ValueError(f'{archive}: incomplete or nested session layout: {candidate.name}')
                schema = json.loads(manifest.read_text()).get('schema_version')
                if type(schema) is int and schema == 1:
                    required = ['metadata.json', 'frames.csv', 'wheel.csv', 'telemetry.csv']
                    image_directory = 'images'
                elif schema == 'record_py_aligned_v1':
                    required = ['metadata.json', 'meta.json', 'labels.csv']
                    image_directory = 'frames'
                else:
                    raise ValueError(f'{archive}: unsupported session schema: {schema!r}')
                if not all((candidate / name).is_file() for name in required) or not (candidate / image_directory).is_dir():
                    raise ValueError(f'{archive}: incomplete or nested session layout: {candidate.name}')
                # Use unique local names; session_id uniqueness is checked by the validator.
                target = combined / f'session-{len(list(combined.iterdir())):05d}'
                shutil.move(str(candidate), target)
        sessions = load_sessions(combined)
        if any(not session.samples for session in sessions):
            raise ValueError('archive contains a session with no eligible frames')
        summaries = [session.summary() for session in sessions]
        combined.rename(destination)
    return summaries


In [ ]:
import io
import subprocess
import sys
import tempfile

if REPO_SOURCE == 'upload_zip':
    from google.colab import files
    uploaded = files.upload()  # Select exactly one private GitHub repository ZIP.
    if len(uploaded) != 1:
        raise ValueError('Select exactly one repository ZIP')
    archive_name, archive_bytes = next(iter(uploaded.items()))
    if not archive_name.lower().endswith('.zip'):
        raise ValueError('Expected a repository ZIP')
    # Unique runtime-local directory makes this setup safe to rerun.
    repo_container = Path(tempfile.mkdtemp(prefix='forza-repo-', dir='/content'))
    REPO = find_repository(extract_zip(io.BytesIO(archive_bytes), repo_container / 'source'))
    del uploaded, archive_bytes
elif REPO_SOURCE == 'existing':
    REPO = find_repository(EXISTING_REPO)
else:
    raise ValueError('REPO_SOURCE must be upload_zip or existing')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO}[training]'], check=True)
# Editable installs add their path at interpreter startup; make it available now.
sys.path.insert(0, str(REPO / 'src'))
print('Repository:', REPO)


For real data, place completed `.zip` sessions in `MyDrive/forza-ai/session-zips`. Each ZIP must contain either one session's files at its root, or one/more session folders directly at its root. No extra wrapper directories or loose files. Each session needs `metadata.json` with `completed: true`. V1 sessions contain `frames.csv`, `wheel.csv`, `telemetry.csv`, and `images/`; imported `record_py_aligned_v1` recordings contain `meta.json`, `labels.csv`, and `frames/`. Run `forza-train import-recording SOURCE DEST --expert-mode manual` before archiving raw recorder output. Both layouts are validated by the same dataset loader after extraction.

The next cell mounts Drive only for real data, extracts into runtime-local disk, and validates every session before publishing `DATA`. Synthetic mode needs neither Drive nor a GPU. To rerun with existing data, select `REUSE_DATA = True`; otherwise choose a new `DATA` directory. Configure `RUN` to a fresh run directory, or the original directory when resuming.


In [ ]:
import torch
from forza_ai.data.synthetic import generate
from forza_ai.training.cli import main

if not SYNTHETIC:
    from google.colab import drive
    drive.mount('/content/drive')
    ARCHIVE_DIR = Path('/content/drive/MyDrive/forza-ai/session-zips')
    RUN = Path('/content/drive/MyDrive/forza-ai/runs/baseline-001')
    if not REUSE_DATA:
        archives = sorted(ARCHIVE_DIR.glob('*.zip'))
        print(extract_sessions(archives, DATA))
else:
    RUN = Path('/content/forza-smoke-run')  # Ephemeral synthetic checkpoints only.
    if not REUSE_DATA:
        generate(DATA, sessions=3, frames=8, seed=7)
    if DEVICE == 'cpu':
        torch.set_num_threads(1)

if DEVICE not in ('cpu', 'cuda'):
    raise ValueError('Choose cpu or cuda explicitly')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise ValueError('Select a GPU runtime, or explicitly choose cpu for smoke testing')
print('Device:', DEVICE, 'Data:', DATA, 'Checkpoints:', RUN)
main(['validate', str(DATA)])


In [ ]:
if RESUME:
    main(['resume', str(RUN / 'last.pt'), str(DATA), '--epochs', str(EPOCHS), '--device', DEVICE])
else:
    main(['train', str(DATA), str(RUN), '--epochs', str(EPOCHS), '--device', DEVICE,
          '--batch-size', str(BATCH_SIZE)])


After interruption, rerun repository setup and restore data on runtime-local disk. Set `RESUME = True`, keep the original `RUN`, and increase `EPOCHS` to the total desired count. Resume verifies exact data fingerprints and restores model, optimizer, configuration and split. Synthetic checkpoints disappear with the runtime; real checkpoints persist in Drive.


In [ ]:
main(['evaluate', str(RUN / 'best.pt'), str(DATA), '--device', DEVICE])
# Unique export folder allows reevaluation after resume without overwriting artifacts.
from uuid import uuid4
EXPORT = RUN / f'export-best-through-epoch-{EPOCHS}-{uuid4().hex[:8]}'
main(['export', str(RUN / 'best.pt'), str(EXPORT)])
print('Portable CPU artifact:', EXPORT)


Download the exported directory for CPU shadow inference. Synthetic success proves software execution only. Offline errors do not establish driving quality. See `docs/TRAINING.md` for the predictor interface and further CLI options.
